# Test Your Algorithm

## Instructions
1. From the **Pulse Rate Algorithm** Notebook you can do one of the following:
   - Copy over all the **Code** section to the following Code block.
   - Download as a Python (`.py`) and copy the code to the following Code block.
2. In the bottom right, click the <span style="color:blue">Test Run</span> button. 

### Didn't Pass
If your code didn't pass the test, go back to the previous Concept or to your local setup and continue iterating on your algorithm and try to bring your training error down before testing again.

### Pass
If your code passes the test, complete the following! You **must** include a screenshot of your code and the Test being **Passed**. Here is what the starter filler code looks like when the test is run and should be similar. A passed test will include in the notebook a green outline plus a box with **Test passed:** and in the Results bar at the bottom the progress bar will be at 100% plus a checkmark with **All cells passed**.
![Example](example.png)

1. Take a screenshot of your code passing the test, make sure it is in the format `.png`. If not a `.png` image, you will have to edit the Markdown render the image after Step 3. Here is an example of what the `passed.png` would look like 
2. Upload the screenshot to the same folder or directory as this jupyter notebook.
3. Rename the screenshot to `passed.png` and it should show up below.
![Passed](passed.png)
4. Download this jupyter notebook as a `.pdf` file. 
5. Continue to Part 2 of the Project. 

In [ ]:
import glob

import numpy as np
import scipy as sp
import scipy.io
import scipy.signal


def LoadTroikaDataset():
    """
    Retrieve the .mat filenames for the troika dataset.

    Review the README in ./datasets/troika/ to understand the organization of the .mat files.

    Returns:
        data_fls: Names of the .mat files that contain signal data
        ref_fls: Names of the .mat files that contain reference data
        <data_fls> and <ref_fls> are ordered correspondingly, so that ref_fls[5] is the 
            reference data for data_fls[5], etc...
    """
    data_dir = "./datasets/troika/training_data"
    data_fls = sorted(glob.glob(data_dir + "/DATA_*.mat"))
    ref_fls = sorted(glob.glob(data_dir + "/REF_*.mat"))
    return data_fls, ref_fls

def LoadTroikaDataFile(data_fl):
    """
    Loads and extracts signals from a troika data file.

    Usage:
        data_fls, ref_fls = LoadTroikaDataset()
        ppg, accx, accy, accz = LoadTroikaDataFile(data_fls[0])

    Args:
        data_fl: (str) filepath to a troika .mat file.

    Returns:
        numpy arrays for ppg, accx, accy, accz signals.
    """
    data = sp.io.loadmat(data_fl)['sig']
    return data[2:]


def AggregateErrorMetric(pr_errors, confidence_est):
    """
    Computes an aggregate error metric based on confidence estimates.

    Computes the MAE at 90% availability. 

    Args:
        pr_errors: a numpy array of errors between pulse rate estimates and corresponding 
            reference heart rates.
        confidence_est: a numpy array of confidence estimates for each pulse rate
            error.

    Returns:
        the MAE at 90% availability
    """
    # Higher confidence means a better estimate. The best 90% of the estimates
    #    are above the 10th percentile confidence.
    percentile90_confidence = np.percentile(confidence_est, 10)

    # Find the errors of the best pulse rate estimates
    best_estimates = pr_errors[confidence_est >= percentile90_confidence]

    # Return the mean absolute error
    return np.mean(np.abs(best_estimates))

def Evaluate():
    """
    Top-level function evaluation function.

    Runs the pulse rate algorithm on the Troika dataset and returns an aggregate error metric.

    Returns:
        Pulse rate error on the Troika dataset. See AggregateErrorMetric.
    """
    # Retrieve dataset files
    data_fls, ref_fls = LoadTroikaDataset()
    errs, confs = [], []
    for data_fl, ref_fl in zip(data_fls, ref_fls):
        # Run the pulse rate algorithm on each trial in the dataset
        errors, confidence = RunPulseRateAlgorithm(data_fl, ref_fl)
        errs.append(errors)
        confs.append(confidence)
        # Compute aggregate error metric
    errs = np.hstack(errs)
    confs = np.hstack(confs)
    return AggregateErrorMetric(errs, confs)

def find_frequency_peaks(fft_mags, fft_freqs, bandpass):
    """
    Find peak indices of frequencies.

    Args:
        fft_mags: a numpy array of FFT magnitudes of a signal
        fft_freqs: a numpy array of the corresponding frequency values
        bandpass: a tuple of frequency bandpass

    Returns:
        peaks: a list of tuples of peaks' magnitude and frequency, sorted by magnitude
    """
    lf, hf = bandpass
    min_height = 1.5 * np.mean(fft_mags[(fft_freqs > lf) & (fft_freqs < hf)]) # Mean of magnitudes within the bandpass
    f_distance = 2 # Corresponding to 0.125 Hz (7.5 BPM) minimum distance between peaks

    # Get peaks
    peak_locs = sp.signal.find_peaks(fft_mags, height=min_height, distance=f_distance)[0]
    peak_mags = fft_mags[peak_locs]
    peak_f = fft_freqs[peak_locs]
    peaks = sorted(list(zip(peak_mags, peak_f)), reverse=True)

    return peaks
    

def RunPulseRateAlgorithm(data_fl, ref_fl):
    """
    Running pulse-rate estimation algorithm on the input data
    and returning error and confidence for each estimations.

    Args:
        data_fl: a string of where the data file is saved
        ref_fl: a string of where the reference file is saved

    Returns:
        errors: a numpy array containing pulse-rate estimates
                for each window of time
        confidence: a numpy array containing each estimate's confidence
    """
    # Load data using LoadTroikaDataFile
    ppg, accx, accy, accz = LoadTroikaDataFile(data_fl)
    ppg -= np.mean(ppg)
    accx -= np.mean(accx)
    accy -= np.mean(accy)
    accz -= np.mean(accz)
    ref = np.hstack(sp.io.loadmat(ref_fl)['BPM0'])
    fs = 125
    
    # 40-240 BPM Bandpass Filter
    low = 40 / 60 # Hz
    high = 240 / 60 # Hz
    band = (low, high)
    b, a = sp.signal.butter(N=3, Wn=band, btype='bandpass', fs=fs)
    ppg = sp.signal.filtfilt(b, a, ppg)
    accx = sp.signal.filtfilt(b, a, accx)
    accy = sp.signal.filtfilt(b, a, accy)
    accz = sp.signal.filtfilt(b, a, accz)
    # accm = np.sqrt(np.sum(np.square(np.vstack((accx, accy, accz))), axis=0)) # Accelerometer magnitude

    # Rolling the window
    preds = list()
    confidences = list()
    n_window = 8 * fs # 8 seconds
    n_sliding = 2 * fs # 2 seconds
    prev_pred = None

    for i in range(0, len(ppg) - n_window + 1, n_sliding):
        window_ppg = ppg[i:i+n_window]
        window_accx = accx[i:i+n_window]
        window_accy = accy[i:i+n_window]
        window_accz = accz[i:i+n_window]
        
        # Obtain frequency domain of signals
        freqs = np.fft.rfftfreq(n_window, d=1/125)
        ppg_fft = abs(np.fft.rfft(window_ppg))
        accx_fft = abs(np.fft.rfft(window_accx))
        accy_fft = abs(np.fft.rfft(window_accy))
        accz_fft = abs(np.fft.rfft(window_accz))

        # Zero-ing frequencies outside bandpass
        ppg_fft[(freqs < low) | (freqs > high)] = 0
        accx_fft[(freqs < low) | (freqs > high)] = 0
        accy_fft[(freqs < low) | (freqs > high)] = 0
        accz_fft[(freqs < low) | (freqs > high)] = 0

        # Get PPG peaks
        ppg_peaks = find_frequency_peaks(ppg_fft, freqs, (low, high))

        # Get prediction
        f_tol = 5 / 60 # 15 BPM tolerance window
        pred = ppg_peaks[0][1] # Default prediction, greatest PPG frequency


        # If PPG peak is similar to accel's and their harmonics, while PPG peaks are more than 1
        peak_accx = freqs[np.argmax(accx_fft)]
        peak_accy = freqs[np.argmax(accy_fft)]
        peak_accz = freqs[np.argmax(accz_fft)]
        
        if ((abs(pred-peak_accx) <= f_tol) or
            (abs(pred-peak_accy) <= f_tol) or
            (abs(pred-peak_accz) <= f_tol)) and len(ppg_peaks) > 1:
            pred = ppg_peaks[1][1]
            ppg_peaks = ppg_peaks[1:] # Remove the frequency from smoothing candidates

        # Smoothing by getting the best candidate that's nearest to the previous predicted pulse-rate
        candidates = [f for _, f in ppg_peaks[:5]]

        if prev_pred is None:
            pred = candidates[0]
        else:
            pred = min(candidates, key=lambda f: abs(f - prev_pred))

        prev_pred = pred        
        pred_bpm = pred * 60 # Convert to beats-per-minute (BPM)
        preds.append(pred_bpm)

        # Get Confidence as signal-to-noise-ratio (SNR)
        f_loc = (freqs > pred-f_tol) & (freqs < pred+f_tol)
        signal_power = ppg_fft[f_loc].sum()
        snr = signal_power / ppg_fft.sum()
        confidences.append(snr)

    # Return per-estimate mean absolute error and confidence as a 2-tuple of numpy arrays.
    errors = abs(preds - ref)
    confidence = np.array(confidences)
    return errors, confidence